# DishDash — Milestone 3: Data Collection & Merging

**Course:** TECH-UB.0024 — Projects in Programming & AI (NYU Stern, Spring 2026)  
**Group:** S2 Altman's Angels  
**Project:** DishDash — Hyperlocal Restaurant Discovery & Food Intelligence

---

## What This Notebook Does

This notebook pulls data from **3 different sources**, merges them into a single
Pandas DataFrame, and stores everything in a **SQLite database**.

| # | Data Source | What It Gives Us | Auth? |
|---|-----------|-------------------|-------|
| 1 | **Overpass API (OpenStreetMap)** | Restaurant locations, cuisine tags, addresses near NYU | None |
| 2 | **Google Places API (New)** | Ratings, price levels, reviews, open hours, categories | API key (free tier) |
| 3 | **Open Food Facts API** | Nutritional data, Nutri-Score, allergens by food type | None |

**Bonus:** We also run **HuggingFace sentiment analysis** on Google review text.

### How the Merge Works
```
Overpass (OSM)    ──┐
                     ├── Fuzzy name + geo proximity ──> Merged Venues
Google Places     ──┘                                       │
                                                             ├── cuisine category ──> Final DataFrame
Open Food Facts   ──────────────────────────────────────────┘
```

---
## 0. Setup & Installs

First, we install the libraries we need and set up our API keys.  
**Everyone in the group can run this cell first.**

In [1]:
# Install libraries not pre-installed in Colab
!pip install thefuzz python-Levenshtein --quiet

print("All packages installed!")

All packages installed!


In [2]:
# ============================================================
# IMPORTS
# ============================================================

import requests
import pandas as pd
import sqlite3
import json
import time
import math
from thefuzz import fuzz

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 40)

print("All imports loaded!")

All imports loaded!


In [ ]:
# ============================================================
# API KEYS
# ============================================================
#
# GOOGLE PLACES API (New) — Setup:
#   1. Go to https://console.cloud.google.com
#   2. Create a project (e.g., "DishDash")
#   3. APIs & Services -> Library -> search "Places API (New)" -> Enable
#   4. APIs & Services -> Credentials -> Create Credentials -> API Key
#   5. Copy the key and paste it below
#
#   You need billing enabled, but the free tier gives you:
#     - Essentials: 10,000 free requests/month
#     - Pro: 5,000 free requests/month
#   A student project will use maybe 50-100 requests total.
#
# HUGGING FACE (optional, for sentiment analysis):
#   - https://huggingface.co -> Sign up -> Settings -> Access Tokens
# ============================================================

GOOGLE_API_KEY    = ""   # <-- paste your Google API key
HUGGINGFACE_TOKEN = ""     # <-- paste your HF token (optional)

# ============================================================
# CONSTANTS — search area (NYU / Greenwich Village)
# ============================================================

NYU_LAT = 40.7295   # Washington Square Park
NYU_LNG = -73.9965

# Bounding box for Overpass (South, West, North, East)
BBOX_SOUTH = 40.7240
BBOX_WEST  = -74.0020
BBOX_NORTH = 40.7380
BBOX_EAST  = -73.9900

SEARCH_RADIUS_M = 800  # ~10 block radius

print(f"Search center: ({NYU_LAT}, {NYU_LNG})")
print("Configuration ready!")

Search center: (40.7295, -73.9965)
Configuration ready!


In [4]:
import requests
resp = requests.post(
    "https://places.googleapis.com/v1/places:searchNearby",
    headers={
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_API_KEY,
        "X-Goog-FieldMask": "places.displayName"
    },
    json={"locationRestriction": {"circle": {"center": {"latitude": 40.7295, "longitude": -73.9965}, "radius": 500}},
          "includedTypes": ["restaurant"], "maxResultCount": 1}
)
print(f"Status: {resp.status_code}")
print(resp.text[:300])

Status: 200
{
  "places": [
    {
      "displayName": {
        "text": "Lafayette Grand Café & Bakery",
        "languageCode": "en"
      }
    }
  ]
}



---
## 1. Data Source #1 — Overpass API (OpenStreetMap)

**What is it?** OpenStreetMap is like Wikipedia for maps — community-edited geographic data.  
The Overpass API lets us query OSM data with filters.

**What we get:** Every restaurant, cafe, and fast food place near NYU with
name, lat/lng, cuisine type, address, phone, website, and hours.

**Auth required?** None!

In [5]:
def fetch_osm_restaurants(bbox_s, bbox_w, bbox_n, bbox_e, max_retries=3):
    """Query the Overpass API with automatic retries on timeout."""

    overpass_query = f"""
    [out:json][timeout:60];
    (
      node["amenity"="restaurant"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
      node["amenity"="cafe"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
      node["amenity"="fast_food"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
    );
    out body;
    """

    OVERPASS_URL = "https://overpass-api.de/api/interpreter"

    for attempt in range(1, max_retries + 1):
        print(f"Querying Overpass API (attempt {attempt}/{max_retries})...")
        try:
            response = requests.get(OVERPASS_URL, params={"data": overpass_query}, timeout=90)

            if response.status_code == 200:
                elements = response.json().get("elements", [])
                print(f"Found {len(elements)} places from OSM")

                records = []
                for el in elements:
                    tags = el.get("tags", {})
                    records.append({
                        "osm_id":        el.get("id"),
                        "name":          tags.get("name", "Unknown"),
                        "lat":           el.get("lat"),
                        "lng":           el.get("lon"),
                        "amenity_type":  tags.get("amenity", ""),
                        "cuisine":       tags.get("cuisine", "unknown"),
                        "addr_street":   tags.get("addr:street", ""),
                        "addr_number":   tags.get("addr:housenumber", ""),
                        "phone":         tags.get("phone", ""),
                        "website":       tags.get("website", ""),
                        "opening_hours": tags.get("opening_hours", ""),
                        "wheelchair":    tags.get("wheelchair", ""),
                    })

                df = pd.DataFrame(records)
                df = df[df["name"] != "Unknown"].reset_index(drop=True)
                print(f"{len(df)} named places after cleaning")
                return df

            print(f"Error: status code {response.status_code}")

        except requests.exceptions.Timeout:
            print(f"Timeout on attempt {attempt}")

        if attempt < max_retries:
            wait = attempt * 10
            print(f"Waiting {wait}s before retry...")
            time.sleep(wait)

    print("All retries failed. Returning empty DataFrame.")
    return pd.DataFrame()

In [6]:
# Fetch OSM data
df_osm = fetch_osm_restaurants(BBOX_SOUTH, BBOX_WEST, BBOX_NORTH, BBOX_EAST)

print(f"\n{'='*60}")
print("OSM DATA SUMMARY")
print(f"{'='*60}")
print(f"Total restaurants: {len(df_osm)}")

if len(df_osm) > 0:
    print(f"\nCuisine breakdown:")
    print(df_osm['cuisine'].value_counts().head(10))
    print(f"\nAmenity types:")
    print(df_osm['amenity_type'].value_counts())
    display(df_osm.head())
else:
    print("\nNo data returned — Overpass may be overloaded.")
    print("Try running this cell again in a minute.")


Querying Overpass API (attempt 1/3)...
Found 395 places from OSM
392 named places after cleaning

OSM DATA SUMMARY
Total restaurants: 392

Cuisine breakdown:
cuisine
unknown        74
coffee_shop    30
italian        25
mexican        16
sandwich       15
pizza          14
japanese       11
burger         10
chinese         9
bubble_tea      7
Name: count, dtype: int64

Amenity types:
amenity_type
restaurant    229
cafe           87
fast_food      76
Name: count, dtype: int64


,osm_id,name,lat,lng,amenity_type,cuisine,addr_street,addr_number,phone,website,opening_hours,wheelchair
0,886175538,Think Coffee,40.725407,-73.992390,cafe,coffee_shop,Bleecker Street,1,,https://www.thinkcoffee.com/,"Mo-Fr 06:30-19:00; Sa,Su 07:00-19:00",yes
1,886682281,Le Basket,40.727976,-73.994851,fast_food,deli,Broadway,683,,,24/7,
2,886702490,Starbucks,40.727207,-73.995481,cafe,coffee_shop,Broadway,665,+1-917-534-0799,,Mo-Su 06:00-21:00,yes
3,1714427541,Shmoné,40.733508,-73.999067,restaurant,israeli,West 8th Steet,61,+1-646-438-9815,https://www.shmonenyc.com,Mo-Sa 18:00-12:00,
4,1883739637,Laut,40.737548,-73.991104,restaurant,thai,East 17th Street,15,+1 212-206-8989,https://www.lautnyc.com/,Mo-Fr 11:30-22:30; Sa 14:00-22:30; S...,yes


---
## 2. Data Source #2 — Google Places API (New)

**What is it?** Google's Places API gives us access to Google Maps venue data —
ratings, reviews, price levels, business status, and more.

**What we get:**
- Business name, location, formatted address
- Google rating (1.0-5.0) and total number of ratings
- Price level (PRICE_LEVEL_INEXPENSIVE to PRICE_LEVEL_VERY_EXPENSIVE)
- Types/categories
- User reviews (text + individual rating)

**Auth required?** Yes — Google Cloud API key with Places API (New) enabled.

**How the new API works:**  
The Places API (New) uses **POST requests** with JSON bodies, and you control
which fields are returned (and billed) using a **FieldMask** header. We request
only the fields we need to stay within the free tier.

**Setup (5 minutes):**
1. Go to https://console.cloud.google.com
2. Create a project -> Enable "Places API (New)"
3. Create an API key under Credentials
4. Paste it in Section 0 above

In [ ]:
def fetch_google_places(lat, lng, radius_m, api_key, max_results=20):
    """
    Search for restaurants near a location using Google Places API (New).

    Uses the Nearby Search endpoint (POST request).
    The FieldMask header controls which fields we get back —
    and what we get billed for. We stick to basic + a few pro fields.

    Parameters:
        lat, lng:      center point coordinates
        radius_m:      search radius in meters
        api_key:       your Google Cloud API key
        max_results:   max results (Google caps at 20 per request)

    Returns:
        pandas DataFrame with venue data
    """

    if api_key == "YOUR_GOOGLE_KEY_HERE":
        print("Please set your Google API key in Section 0!")
        return pd.DataFrame()

    url = "https://places.googleapis.com/v1/places:searchNearby"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        # FieldMask: controls which fields are returned
        # displayName, location, formattedAddress = Essentials (free 10K/mo)
        # rating, userRatingCount, priceLevel, types = Pro (free 5K/mo)
        # reviews = Pro
        "X-Goog-FieldMask": ",".join([
            "places.id",
            "places.displayName",
            "places.location",
            "places.formattedAddress",
            "places.types",
            "places.rating",
            "places.userRatingCount",
            "places.priceLevel",
            "places.reviews",
            "places.businessStatus",
        ])
    }

    # Request body for Nearby Search
    body = {
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": lat,
                    "longitude": lng
                },
                "radius": radius_m
            }
        },
        "includedTypes": ["restaurant", "cafe", "fast_food_restaurant",
                          "pizza_restaurant", "coffee_shop", "meal_takeaway"],
        "maxResultCount": max_results,
        "languageCode": "en",
    }

    print("Querying Google Places API (New)...")
    response = requests.post(url, headers=headers, json=body)

    if response.status_code != 200:
        print(f"Error: status code {response.status_code}")
        error_data = response.json() if response.text else {}
        error_msg = error_data.get("error", {}).get("message", response.text[:300])
        print(f"Message: {error_msg}")
        return pd.DataFrame()

    data = response.json()
    places = data.get("places", [])
    print(f"Found {len(places)} places from Google")

    records = []
    for place in places:
        # Extract display name
        display_name = place.get("displayName", {}).get("text", "Unknown")

        # Extract location
        loc = place.get("location", {})

        # Extract first review text (if any)
        reviews = place.get("reviews", [])
        review_texts = []
        for r in reviews[:3]:  # up to 3 reviews
            txt = r.get("text", {}).get("text", "")
            if txt:
                review_texts.append(txt)
        combined_reviews = " | ".join(review_texts)

        # Map Google's price level enum to a readable format
        price_map = {
            "PRICE_LEVEL_FREE": "Free",
            "PRICE_LEVEL_INEXPENSIVE": "$",
            "PRICE_LEVEL_MODERATE": "$$",
            "PRICE_LEVEL_EXPENSIVE": "$$$",
            "PRICE_LEVEL_VERY_EXPENSIVE": "$$$$",
        }
        raw_price = place.get("priceLevel", "")
        price_level = price_map.get(raw_price, raw_price)

        # Get types (categories)
        types = place.get("types", [])
        # Filter out generic types to find the most useful one
        useful_types = [t for t in types if t not in
                        ("point_of_interest", "establishment", "food")]
        primary_type = useful_types[0] if useful_types else (types[0] if types else "unknown")

        records.append({
            "google_place_id":  place.get("id", ""),
            "name":             display_name,
            "lat":              loc.get("latitude"),
            "lng":              loc.get("longitude"),
            "address":          place.get("formattedAddress", ""),
            "google_type":      primary_type,
            "all_types":        ", ".join(types[:5]),
            "price_level":      price_level,
            "google_rating":    place.get("rating"),
            "rating_count":     place.get("userRatingCount", 0),
            "business_status":  place.get("businessStatus", ""),
            "review_text":      combined_reviews,
        })

    df = pd.DataFrame(records)
    df = df[df["name"] != "Unknown"].reset_index(drop=True)
    print(f"{len(df)} named venues after cleaning")
    return df

In [8]:
# Call Google Places API
df_google = fetch_google_places(NYU_LAT, NYU_LNG, SEARCH_RADIUS_M, GOOGLE_API_KEY)

if len(df_google) > 0:
    print(f"\n{'='*60}")
    print("GOOGLE PLACES DATA SUMMARY")
    print(f"{'='*60}")
    print(f"Total venues: {len(df_google)}")
    print(f"\nType breakdown:")
    print(df_google['google_type'].value_counts().head(10))
    print(f"\nPrice level distribution:")
    print(df_google['price_level'].value_counts())
    print(f"\nAvg rating: {df_google['google_rating'].mean():.1f}")
    print(f"Venues with reviews: {df_google['review_text'].astype(bool).sum()}")
    display(df_google.head())
else:
    print("\nNo Google data — did you set your API key?")

Querying Google Places API (New)...
Found 20 places from Google
20 named venues after cleaning

GOOGLE PLACES DATA SUMMARY
Total venues: 20

Type breakdown:
google_type
pizza_restaurant        5
french_restaurant       3
thai_restaurant         2
movie_theater           1
hamburger_restaurant    1
coffee_shop             1
bakery                  1
cafe                    1
fast_food_restaurant    1
italian_restaurant      1
Name: count, dtype: int64

Price level distribution:
price_level
$$     6
$$$    5
       5
$      4
Name: count, dtype: int64

Avg rating: 4.4
Venues with reviews: 20


,google_place_id,name,lat,lng,address,google_type,all_types,price_level,google_rating,rating_count,business_status,review_text
0,ChIJuW43oZNZwokRdE5tLzpuykE,John's of Bleecker Street,40.731619,-74.003447,"278 Bleecker St, New York, NY 10014,...",pizza_restaurant,"pizza_restaurant, meal_takeaway, res...",$$,4.6,8093,OPERATIONAL,Really great little slice. We got a ...
1,ChIJt7fMLIlZwokRCRtM9bNDg78,Balthazar,40.722668,-73.998230,"80 Spring St, New York, NY 10012, USA",french_restaurant,"french_restaurant, seafood_restauran...",$$$,4.4,7635,OPERATIONAL,A very pleasant experience. The snai...
2,ChIJBVtTt5pZwokR48CgdYqHma8,Lafayette Grand Café & Bakery,40.727593,-73.993702,"380 Lafayette St, New York, NY 10003...",french_restaurant,"french_restaurant, pastry_shop, dess...",$$$,4.2,5401,OPERATIONAL,A Must-Visit Gem in NoHo! ⭐⭐⭐⭐⭐\nIf ...
3,ChIJQ1RTt49ZwokRgYgqG6DdK2w,Angelika Film Center & Cafe - New York,40.725907,-73.997169,"18 W Houston St, New York, NY 10012,...",movie_theater,"movie_theater, cafe, event_venue, fo...",,4.4,2184,OPERATIONAL,Very cute theatre!!! Came here for a...
4,ChIJs8MdNo9ZwokRTPUHiArLC-o,Rubirosa,40.722738,-73.996135,"235 Mulberry St, New York, NY 10012,...",pizza_restaurant,"pizza_restaurant, food_store, meal_t...",$$,4.6,6708,OPERATIONAL,"Fun, quintessential little Italy vib..."


In [9]:
# FALLBACK: If OSM returned nothing, use Google as the base instead
if len(df_osm) == 0 and len(df_google) > 0:
    print("OSM returned no data — using Google Places as the base instead.\n")
    df_osm = df_google.rename(columns={
        "google_place_id": "osm_id",
        "google_type": "amenity_type",
        "address": "addr_street",
    }).copy()
    df_osm['cuisine'] = df_osm.get('all_types', 'unknown')
    df_osm['addr_number'] = ''
    df_osm['phone'] = ''
    df_osm['website'] = ''
    df_osm['opening_hours'] = ''
    df_osm['wheelchair'] = ''
    print(f"Using {len(df_osm)} Google places as base DataFrame")
else:
    print(f"OSM has {len(df_osm)} places — proceeding normally")


OSM has 392 places — proceeding normally


---
## 3. Data Source #3 — Open Food Facts API

**What is it?** Open Food Facts is a free, open database of food products.
Think of it as Wikipedia for nutrition labels.

**What we get:** Nutritional info (calories, fat, protein, carbs, sugar, sodium),
Nutri-Score (A-E health grade), and allergen warnings.

**Our approach:** We can't look up a *restaurant* on Open Food Facts — it tracks
*products*, not venues. So we map each restaurant's **cuisine type** to a
representative food search, then compute average nutrition stats per cuisine.

**Auth required?** None!

In [10]:
# ============================================================
# CUISINE -> FOOD SEARCH MAPPING
# ============================================================

CUISINE_TO_SEARCH = {
    "pizza":       ["pizza"],
    "italian":     ["pasta", "pizza", "risotto"],
    "chinese":     ["noodles", "fried rice", "dumplings"],
    "japanese":    ["sushi", "ramen", "miso"],
    "sushi":       ["sushi", "sashimi"],
    "mexican":     ["burrito", "taco", "nachos"],
    "burger":      ["burger", "hamburger"],
    "american":    ["burger", "sandwich", "fries"],
    "indian":      ["curry", "naan", "biryani"],
    "thai":        ["pad thai", "curry", "spring roll"],
    "mediterranean":["hummus", "falafel", "pita"],
    "french":      ["croissant", "baguette", "quiche"],
    "korean":      ["bibimbap", "kimchi", "bulgogi"],
    "vietnamese":  ["pho", "banh mi", "spring roll"],
    "sandwich":    ["sandwich", "sub"],
    "coffee_shop": ["coffee", "latte", "muffin"],
    "cafe":        ["coffee", "pastry", "sandwich"],
    "unknown":     ["meal", "food"],
}

print(f"Defined search terms for {len(CUISINE_TO_SEARCH)} cuisine types")

Defined search terms for 18 cuisine types


In [11]:
def fetch_nutrition_for_cuisine(cuisine_key, search_terms, max_products=5):
    BASE_URL = "https://world.openfoodfacts.org/cgi/search.pl"
    all_nutrition = []

    for term in search_terms:
        params = {
            "search_terms": term, "json": 1, "page_size": max_products,
            "fields": "product_name,nutriscore_grade,nutriments,allergens_tags",
        }

        # Try up to 2 attempts per term with longer timeout
        for attempt in range(2):
            try:
                resp = requests.get(BASE_URL, params=params, timeout=30)
                if resp.status_code != 200:
                    break

                for p in resp.json().get("products", []):
                    nuts = p.get("nutriments", {})
                    if nuts.get("energy-kcal_100g") is not None:
                        all_nutrition.append({
                            "calories_100g": nuts.get("energy-kcal_100g", 0),
                            "fat_100g":      nuts.get("fat_100g", 0),
                            "protein_100g":  nuts.get("proteins_100g", 0),
                            "carbs_100g":    nuts.get("carbohydrates_100g", 0),
                            "sugar_100g":    nuts.get("sugars_100g", 0),
                            "sodium_100g":   nuts.get("sodium_100g", 0),
                            "nutriscore":    p.get("nutriscore_grade", ""),
                        })
                break  # Success — don't retry

            except requests.exceptions.RequestException as e:
                if attempt == 0:
                    print(f"    Retrying '{term}' in 5s...")
                    time.sleep(5)
                else:
                    print(f"    Skipped '{term}' (timeout)")

        time.sleep(1)  # Longer pause between terms

    if not all_nutrition:
        return {
            "cuisine_key": cuisine_key,
            "avg_calories_100g": None, "avg_fat_100g": None,
            "avg_protein_100g": None, "avg_carbs_100g": None,
            "avg_sugar_100g": None, "avg_sodium_100g": None,
            "common_nutriscore": None, "off_sample_size": 0
        }

    df_temp = pd.DataFrame(all_nutrition)
    nutri_scores = df_temp["nutriscore"].replace("", pd.NA).dropna()
    common_ns = nutri_scores.mode().iloc[0] if len(nutri_scores) > 0 else None

    return {
        "cuisine_key":       cuisine_key,
        "avg_calories_100g": round(df_temp["calories_100g"].mean(), 1),
        "avg_fat_100g":      round(df_temp["fat_100g"].mean(), 1),
        "avg_protein_100g":  round(df_temp["protein_100g"].mean(), 1),
        "avg_carbs_100g":    round(df_temp["carbs_100g"].mean(), 1),
        "avg_sugar_100g":    round(df_temp["sugar_100g"].mean(), 1),
        "avg_sodium_100g":   round(df_temp["sodium_100g"].mean(), 3),
        "common_nutriscore": common_ns,
        "off_sample_size":   len(all_nutrition),
    }

In [12]:
# Fetch nutrition data for ALL cuisine types
print("Fetching nutritional data from Open Food Facts...")
print("(This takes ~2 min — we add delays to be polite to the API)\n")

nutrition_records = []
for cuisine, terms in CUISINE_TO_SEARCH.items():
    print(f"  Searching: {cuisine} -> {terms}")
    result = fetch_nutrition_for_cuisine(cuisine, terms)
    nutrition_records.append(result)
    time.sleep(1)

df_nutrition = pd.DataFrame(nutrition_records)

print(f"\n{'='*60}")
print("OPEN FOOD FACTS SUMMARY")
print(f"{'='*60}")
print(f"Cuisine profiles collected: {len(df_nutrition)}")
print(f"Cuisines with data: {df_nutrition['avg_calories_100g'].notna().sum()}")
print()
df_nutrition


Fetching nutritional data from Open Food Facts...
(This takes ~2 min — we add delays to be polite to the API)

  Searching: pizza -> ['pizza']
  Searching: italian -> ['pasta', 'pizza', 'risotto']
  Searching: chinese -> ['noodles', 'fried rice', 'dumplings']
  Searching: japanese -> ['sushi', 'ramen', 'miso']
  Searching: sushi -> ['sushi', 'sashimi']
  Searching: mexican -> ['burrito', 'taco', 'nachos']
  Searching: burger -> ['burger', 'hamburger']
  Searching: american -> ['burger', 'sandwich', 'fries']
  Searching: indian -> ['curry', 'naan', 'biryani']
  Searching: thai -> ['pad thai', 'curry', 'spring roll']
  Searching: mediterranean -> ['hummus', 'falafel', 'pita']
  Searching: french -> ['croissant', 'baguette', 'quiche']
  Searching: korean -> ['bibimbap', 'kimchi', 'bulgogi']
  Searching: vietnamese -> ['pho', 'banh mi', 'spring roll']
  Searching: sandwich -> ['sandwich', 'sub']
  Searching: coffee_shop -> ['coffee', 'latte', 'muffin']
  Searching: cafe -> ['coffee', 'past

,cuisine_key,avg_calories_100g,avg_fat_100g,avg_protein_100g,avg_carbs_100g,avg_sugar_100g,avg_sodium_100g,common_nutriscore,off_sample_size
0,pizza,213.8,4.0,6.6,35.6,4.8,0.541,c,5
1,italian,205.0,3.9,7.3,31.5,3.6,0.301,c,15
2,chinese,233.3,6.4,6.3,34.5,1.7,0.435,b,15
3,japanese,239.5,6.8,8.6,30.6,3.9,2.217,unknown,14
4,sushi,226.3,3.6,17.0,28.6,12.9,3.132,c,10
5,mexican,347.5,15.2,7.2,43.9,3.7,0.618,c,13
6,burger,257.6,9.7,11.0,29.4,5.6,0.553,c,9
7,american,367.2,15.2,8.8,46.2,7.1,0.421,c,14
8,indian,219.5,6.9,6.1,31.8,3.4,0.321,c,14
9,thai,150.4,4.5,4.9,21.0,2.6,0.397,a,13


---
## 4. Merging the Data Sources

Now the key part — combining our 3 data sources into one unified DataFrame.

### Merge Strategy

**Step A: OSM + Google Places (fuzzy geo-match)**  
Both sources have restaurant names and coordinates. We match them using:
1. **Geographic proximity** — within ~75 meters of each other?
2. **Name similarity** — fuzzy string matching on names

LEFT JOIN approach: keep ALL OSM restaurants, even without a Google match.

**Step B: Merged venues + Nutrition (cuisine category)**  
LEFT JOIN on the normalized `cuisine_key` — every "pizza" restaurant gets
the same nutrition profile.

In [13]:
def haversine_meters(lat1, lng1, lat2, lng2):
    """Distance in meters between two lat/lng points (Haversine formula)."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lng2 - lng1)
    a = (math.sin(dphi / 2) ** 2 +
         math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Quick test
d = haversine_meters(40.7295, -73.9965, 40.7300, -73.9960)
print(f"Test distance: {d:.1f} meters (should be ~70m)")

Test distance: 69.8 meters (should be ~70m)


In [14]:
def fuzzy_geo_merge(df_osm, df_google, max_distance_m=75, min_name_score=55):
    """
    Match OSM restaurants to Google Places using:
    1. Geographic proximity (within max_distance_m meters)
    2. Fuzzy name matching (score >= min_name_score out of 100)

    LEFT JOIN: all OSM rows are kept, Google data added where matched.
    """

    google_cols = ['google_place_id', 'google_type', 'all_types', 'price_level',
                   'google_rating', 'rating_count', 'review_text',
                   'match_score', 'match_distance_m']

    if len(df_google) == 0:
        print("No Google data — returning OSM data only.")
        for col in google_cols:
            df_osm[col] = None
        return df_osm

    print(f"Fuzzy matching {len(df_osm)} OSM places with {len(df_google)} Google venues...")

    matches = []
    match_count = 0

    for _, osm_row in df_osm.iterrows():
        best_match = None
        best_score = 0

        for _, g_row in df_google.iterrows():
            # Step 1: Geographic distance
            dist = haversine_meters(
                osm_row['lat'], osm_row['lng'],
                g_row['lat'], g_row['lng']
            )
            if dist > max_distance_m:
                continue

            # Step 2: Fuzzy name matching
            name_score = fuzz.token_sort_ratio(
                osm_row['name'].lower(),
                g_row['name'].lower()
            )
            if name_score < min_name_score:
                continue

            # Combined score: name similarity weighted more than distance
            combined_score = name_score * 0.7 + (1 - dist / max_distance_m) * 30

            if combined_score > best_score:
                best_score = combined_score
                best_match = {
                    "google_place_id": g_row['google_place_id'],
                    "google_type":     g_row['google_type'],
                    "all_types":       g_row['all_types'],
                    "price_level":     g_row['price_level'],
                    "google_rating":   g_row['google_rating'],
                    "rating_count":    g_row['rating_count'],
                    "review_text":     g_row.get('review_text', ''),
                    "match_score":     round(combined_score, 1),
                    "match_distance_m": round(dist, 1),
                }

        if best_match:
            matches.append(best_match)
            match_count += 1
        else:
            matches.append({col: None for col in google_cols})

    df_matches = pd.DataFrame(matches)
    df_merged = pd.concat([df_osm.reset_index(drop=True), df_matches], axis=1)

    print(f"Matched {match_count}/{len(df_osm)} OSM places to Google venues")
    print(f"({len(df_osm) - match_count} OSM-only, no Google match)")
    return df_merged

In [15]:
# Run the OSM + Google merge
df_venues = fuzzy_geo_merge(df_osm, df_google)

print(f"\nMerged venue DataFrame: {df_venues.shape[0]} rows x {df_venues.shape[1]} columns")
df_venues.head()

Fuzzy matching 392 OSM places with 20 Google venues...
Matched 5/392 OSM places to Google venues
(387 OSM-only, no Google match)

Merged venue DataFrame: 392 rows x 21 columns


,osm_id,name,lat,lng,amenity_type,cuisine,addr_street,addr_number,phone,website,opening_hours,wheelchair,google_place_id,google_type,all_types,price_level,google_rating,rating_count,review_text,match_score,match_distance_m
0,886175538,Think Coffee,40.725407,-73.992390,cafe,coffee_shop,Bleecker Street,1,,https://www.thinkcoffee.com/,"Mo-Fr 06:30-19:00; Sa,Su 07:00-19:00",yes,None,None,None,None,NaN,NaN,None,NaN,NaN
1,886682281,Le Basket,40.727976,-73.994851,fast_food,deli,Broadway,683,,,24/7,,None,None,None,None,NaN,NaN,None,NaN,NaN
2,886702490,Starbucks,40.727207,-73.995481,cafe,coffee_shop,Broadway,665,+1-917-534-0799,,Mo-Su 06:00-21:00,yes,None,None,None,None,NaN,NaN,None,NaN,NaN
3,1714427541,Shmoné,40.733508,-73.999067,restaurant,israeli,West 8th Steet,61,+1-646-438-9815,https://www.shmonenyc.com,Mo-Sa 18:00-12:00,,None,None,None,None,NaN,NaN,None,NaN,NaN
4,1883739637,Laut,40.737548,-73.991104,restaurant,thai,East 17th Street,15,+1 212-206-8989,https://www.lautnyc.com/,Mo-Fr 11:30-22:30; Sa 14:00-22:30; S...,yes,None,None,None,None,NaN,NaN,None,NaN,NaN


In [16]:
# ============================================================
# Normalize cuisine keys for the nutrition merge
# ============================================================

def normalize_cuisine(cuisine_raw):
    """Normalize OSM cuisine tag into a key matching CUISINE_TO_SEARCH."""
    if not cuisine_raw or cuisine_raw == "unknown":
        return "unknown"

    primary = cuisine_raw.split(";")[0].strip().lower()
    primary = primary.replace("-", "_").replace(" ", "_")

    if primary in CUISINE_TO_SEARCH:
        return primary

    aliases = {
        "coffee": "coffee_shop", "espresso": "coffee_shop",
        "sushi": "sushi", "ramen": "japanese",
        "tacos": "mexican", "burritos": "mexican",
        "kebab": "mediterranean", "falafel": "mediterranean",
        "pho": "vietnamese", "noodle": "chinese",
    }
    for alias, mapped in aliases.items():
        if alias in primary:
            return mapped
    return "unknown"

df_venues["cuisine_key"] = df_venues["cuisine"].apply(normalize_cuisine)

print("Normalized cuisine distribution:")
print(df_venues["cuisine_key"].value_counts())

Normalized cuisine distribution:
cuisine_key
unknown          177
italian           39
coffee_shop       36
sandwich          19
mexican           18
japanese          17
pizza             15
american          12
chinese           11
burger            11
sushi              7
mediterranean      7
indian             7
thai               4
vietnamese         4
korean             4
french             3
cafe               1
Name: count, dtype: int64


In [17]:
# ============================================================
# Merge venues with nutrition data (LEFT JOIN on cuisine_key)
# ============================================================

print(f"Venues shape BEFORE nutrition merge: {df_venues.shape}")

df_final = df_venues.merge(
    df_nutrition, on="cuisine_key", how="left", suffixes=('', '_nutrition')
)

print(f"Final shape AFTER nutrition merge: {df_final.shape}")
print(f"\nMERGE COMPLETE!")
print(f"{df_final.shape[0]} restaurants x {df_final.shape[1]} features")
print(f"Venues with nutrition data: {df_final['avg_calories_100g'].notna().sum()}")

df_final.head()

Venues shape BEFORE nutrition merge: (392, 22)
Final shape AFTER nutrition merge: (392, 30)

MERGE COMPLETE!
392 restaurants x 30 features
Venues with nutrition data: 392


,osm_id,name,lat,lng,amenity_type,cuisine,addr_street,addr_number,phone,website,opening_hours,wheelchair,google_place_id,google_type,all_types,price_level,google_rating,rating_count,review_text,match_score,match_distance_m,cuisine_key,avg_calories_100g,avg_fat_100g,avg_protein_100g,avg_carbs_100g,avg_sugar_100g,avg_sodium_100g,common_nutriscore,off_sample_size
0,886175538,Think Coffee,40.725407,-73.992390,cafe,coffee_shop,Bleecker Street,1,,https://www.thinkcoffee.com/,"Mo-Fr 06:30-19:00; Sa,Su 07:00-19:00",yes,None,None,None,None,NaN,NaN,None,NaN,NaN,coffee_shop,243.3,9.1,7.9,25.2,12.1,0.192,c,12
1,886682281,Le Basket,40.727976,-73.994851,fast_food,deli,Broadway,683,,,24/7,,None,None,None,None,NaN,NaN,None,NaN,NaN,unknown,205.1,10.8,9.7,18.8,2.6,0.191,b,10
2,886702490,Starbucks,40.727207,-73.995481,cafe,coffee_shop,Broadway,665,+1-917-534-0799,,Mo-Su 06:00-21:00,yes,None,None,None,None,NaN,NaN,None,NaN,NaN,coffee_shop,243.3,9.1,7.9,25.2,12.1,0.192,c,12
3,1714427541,Shmoné,40.733508,-73.999067,restaurant,israeli,West 8th Steet,61,+1-646-438-9815,https://www.shmonenyc.com,Mo-Sa 18:00-12:00,,None,None,None,None,NaN,NaN,None,NaN,NaN,unknown,205.1,10.8,9.7,18.8,2.6,0.191,b,10
4,1883739637,Laut,40.737548,-73.991104,restaurant,thai,East 17th Street,15,+1 212-206-8989,https://www.lautnyc.com/,Mo-Fr 11:30-22:30; Sa 14:00-22:30; S...,yes,None,None,None,None,NaN,NaN,None,NaN,NaN,thai,150.4,4.5,4.9,21.0,2.6,0.397,a,13


---
## 5. (Bonus) Sentiment Analysis on Reviews — HuggingFace

For venues that have Google review text, we run NLP sentiment analysis
using HuggingFace's free Inference API.

**Optional for Milestone 3** — gives you a head start on M4/M5.

In [18]:
def analyze_sentiment_hf(text, hf_token):
    """Run sentiment analysis using HuggingFace Inference API."""
    if not text or text.strip() == "":
        return {"sentiment_label": None, "sentiment_score": None}

    API_URL = "https://api-inference.huggingface.co/models/distilbert-base-uncased-finetuned-sst-2-english"
    headers = {"Authorization": f"Bearer {hf_token}"}

    try:
        response = requests.post(API_URL, headers=headers,
                                 json={"inputs": text[:512]}, timeout=15)
        if response.status_code == 200:
            result = response.json()
            if result and isinstance(result, list) and len(result) > 0:
                top = result[0][0] if isinstance(result[0], list) else result[0]
                return {
                    "sentiment_label": top.get("label"),
                    "sentiment_score": round(top.get("score", 0), 3),
                }
    except Exception as e:
        print(f"  Sentiment API error: {e}")

    return {"sentiment_label": None, "sentiment_score": None}


if HUGGINGFACE_TOKEN != "YOUR_HF_TOKEN_HERE":
    review_count = df_final["review_text"].fillna("").str.strip().astype(bool).sum()
    print(f"Running sentiment analysis on {review_count} reviews...")

    sentiments = []
    for _, row in df_final.iterrows():
        text = row.get("review_text", "")
        if text and str(text).strip():
            result = analyze_sentiment_hf(str(text), HUGGINGFACE_TOKEN)
            time.sleep(0.3)
        else:
            result = {"sentiment_label": None, "sentiment_score": None}
        sentiments.append(result)

    df_sentiment = pd.DataFrame(sentiments)
    df_final["sentiment_label"] = df_sentiment["sentiment_label"]
    df_final["sentiment_score"] = df_sentiment["sentiment_score"]
    print(f"\nSentiment analysis complete!")
    print(df_final["sentiment_label"].value_counts())
else:
    print("Skipping sentiment analysis (no HuggingFace token set)")
    df_final["sentiment_label"] = None
    df_final["sentiment_score"] = None

Running sentiment analysis on 5 reviews...

Sentiment analysis complete!
Series([], Name: count, dtype: int64)


---
## 6. Explore the Final Merged DataFrame

In [19]:
print("=" * 70)
print("         DISHDASH — FINAL MERGED DATASET SUMMARY")
print("=" * 70)
print(f"\nShape: {df_final.shape[0]} restaurants x {df_final.shape[1]} columns\n")

print("ALL COLUMNS:")
print("-" * 50)
for i, col in enumerate(df_final.columns):
    source = "[OSM]" if col in ['osm_id','amenity_type','cuisine','addr_street',
                                 'addr_number','phone','website','opening_hours',
                                 'wheelchair'] else \
             "[GOOG]" if col in ['google_place_id','google_type','all_types',
                                  'price_level','google_rating','rating_count',
                                  'review_text','match_score','match_distance_m',
                                  'business_status'] else \
             "[OFF]" if col.startswith('avg_') or col in ['common_nutriscore',
                                                          'off_sample_size'] else \
             "[NLP]" if 'sentiment' in col else \
             "[shared]"
    non_null = df_final[col].notna().sum()
    print(f"  {i+1:2d}. {source:8s} {col:25s} ({non_null}/{len(df_final)} non-null)")

print(f"\nDATA SOURCE COVERAGE:")
print(f"   OSM restaurants:       {len(df_final)}")
print(f"   + Google Places match: {df_final['google_place_id'].notna().sum()}")
print(f"   + Nutrition data:      {df_final['avg_calories_100g'].notna().sum()}")
print(f"   + Sentiment scores:    {df_final['sentiment_label'].notna().sum()}")

         DISHDASH — FINAL MERGED DATASET SUMMARY

Shape: 392 restaurants x 32 columns

ALL COLUMNS:
--------------------------------------------------
   1. [OSM]    osm_id                    (392/392 non-null)
   2. [shared] name                      (392/392 non-null)
   3. [shared] lat                       (392/392 non-null)
   4. [shared] lng                       (392/392 non-null)
   5. [OSM]    amenity_type              (392/392 non-null)
   6. [OSM]    cuisine                   (392/392 non-null)
   7. [OSM]    addr_street               (392/392 non-null)
   8. [OSM]    addr_number               (392/392 non-null)
   9. [OSM]    phone                     (392/392 non-null)
  10. [OSM]    website                   (392/392 non-null)
  11. [OSM]    opening_hours             (392/392 non-null)
  12. [OSM]    wheelchair                (392/392 non-null)
  13. [GOOG]   google_place_id           (5/392 non-null)
  14. [GOOG]   google_type               (5/392 non-null)
  15. [GOOG] 

In [20]:
# Sample view of key columns
cols_to_show = ['name', 'cuisine_key', 'lat', 'lng', 'google_type',
                'google_rating', 'price_level', 'avg_calories_100g',
                'common_nutriscore', 'sentiment_label']
cols_to_show = [c for c in cols_to_show if c in df_final.columns]

print("SAMPLE DATA (key columns):")
df_final[cols_to_show].head(10)

SAMPLE DATA (key columns):


,name,cuisine_key,lat,lng,google_type,google_rating,price_level,avg_calories_100g,common_nutriscore,sentiment_label
0,Think Coffee,coffee_shop,40.725407,-73.992390,None,NaN,None,243.3,c,None
1,Le Basket,unknown,40.727976,-73.994851,None,NaN,None,205.1,b,None
2,Starbucks,coffee_shop,40.727207,-73.995481,None,NaN,None,243.3,c,None
3,Shmoné,unknown,40.733508,-73.999067,None,NaN,None,205.1,b,None
4,Laut,thai,40.737548,-73.991104,None,NaN,None,150.4,a,None
5,Whole Green,unknown,40.737584,-74.000144,None,NaN,None,205.1,b,None
6,Alice,unknown,40.737085,-73.998633,None,NaN,None,205.1,b,None
7,Bite,sandwich,40.725584,-73.994740,None,NaN,None,318.6,d,None
8,maman,unknown,40.732896,-73.993479,None,NaN,None,205.1,b,None
9,Pret A Manger,sandwich,40.737290,-73.990456,None,NaN,None,318.6,d,None


---
## 7. Store in SQLite Database

SQLite is a lightweight database in a single file. We create 3 tables:
1. `venues` — main merged table
2. `nutrition_profiles` — cuisine-level nutrition lookup
3. `data_sources` — metadata documenting our data sources

In [21]:
DB_FILENAME = "dishdash.db"
conn = sqlite3.connect(DB_FILENAME)
print(f"Created SQLite database: {DB_FILENAME}")

# Table 1: venues
df_final.to_sql(name='venues', con=conn, if_exists='replace', index=False)
print(f"Table 'venues': {len(df_final)} rows written")

# Table 2: nutrition_profiles
df_nutrition.to_sql(name='nutrition_profiles', con=conn, if_exists='replace', index=False)
print(f"Table 'nutrition_profiles': {len(df_nutrition)} rows written")

# Table 3: data_sources
data_sources = pd.DataFrame([
    {"source_name": "OpenStreetMap (Overpass API)",
     "url": "https://overpass-api.de/api/interpreter",
     "data_type": "Geospatial restaurant POI data",
     "auth_required": "No",
     "records_fetched": len(df_osm)},
    {"source_name": "Google Places API (New)",
     "url": "https://places.googleapis.com/v1/places:searchNearby",
     "data_type": "Ratings, reviews, price levels, categories",
     "auth_required": "Yes (API key, free tier)",
     "records_fetched": len(df_google)},
    {"source_name": "Open Food Facts",
     "url": "https://world.openfoodfacts.org/cgi/search.pl",
     "data_type": "Nutritional data by food category",
     "auth_required": "No",
     "records_fetched": len(df_nutrition)},
])

data_sources.to_sql(name='data_sources', con=conn, if_exists='replace', index=False)
print(f"Table 'data_sources': {len(data_sources)} rows written")

conn.commit()
print(f"\nAll data committed to {DB_FILENAME}")

Created SQLite database: dishdash.db
Table 'venues': 392 rows written
Table 'nutrition_profiles': 18 rows written
Table 'data_sources': 3 rows written

All data committed to dishdash.db


In [22]:
# ============================================================
# Verify the database
# ============================================================

print("VERIFYING DATABASE CONTENTS\n")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables in database: {list(tables['name'])}\n")

for table_name in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table_name}", conn)
    print(f"  {table_name}: {count['cnt'].iloc[0]} rows")

print("\n" + "=" * 60)
print("SAMPLE QUERY: Top-rated restaurants with nutrition data")
print("=" * 60)

query = """
SELECT name, cuisine_key, google_rating, price_level,
       avg_calories_100g, common_nutriscore
FROM venues
WHERE google_rating IS NOT NULL
  AND avg_calories_100g IS NOT NULL
ORDER BY google_rating DESC
LIMIT 10
"""

pd.read_sql(query, conn)

VERIFYING DATABASE CONTENTS

Tables in database: ['venues', 'nutrition_profiles', 'data_sources']

  venues: 392 rows
  nutrition_profiles: 18 rows
  data_sources: 3 rows

SAMPLE QUERY: Top-rated restaurants with nutrition data


,name,cuisine_key,google_rating,price_level,avg_calories_100g,common_nutriscore
0,Pranakhon,thai,4.7,$$,150.4,a
1,La Cabra,unknown,4.5,,205.1,b
2,12 Matcha,unknown,4.4,,205.1,b
3,Raising Cane's,unknown,4.3,$,205.1,b
4,Lafayette Grand Café & Bakery,french,4.2,$$$,290.2,c


In [23]:
# Average nutrition by cuisine
print("SAMPLE QUERY: Average calories by cuisine")
print("=" * 60)

query2 = """
SELECT cuisine_key,
       COUNT(*) as num_restaurants,
       ROUND(AVG(avg_calories_100g), 1) as avg_cal,
       common_nutriscore
FROM venues
WHERE cuisine_key != 'unknown'
  AND avg_calories_100g IS NOT NULL
GROUP BY cuisine_key
ORDER BY avg_cal DESC
"""

pd.read_sql(query2, conn)

SAMPLE QUERY: Average calories by cuisine


,cuisine_key,num_restaurants,avg_cal,common_nutriscore
0,american,12,367.2,c
1,mexican,18,347.5,c
2,cafe,1,319.3,unknown
3,sandwich,19,318.6,d
4,french,3,290.2,c
5,burger,11,257.6,c
6,coffee_shop,36,243.3,c
7,mediterranean,7,240.0,a
8,japanese,17,239.5,unknown
9,chinese,11,233.3,b


In [24]:
conn.close()
print("Database connection closed.")
print(f"\nYour database file '{DB_FILENAME}' is saved in this Colab session.")
print("To download: click the folder icon (left sidebar) -> find dishdash.db -> download")

Database connection closed.

Your database file 'dishdash.db' is saved in this Colab session.
To download: click the folder icon (left sidebar) -> find dishdash.db -> download


---
## 8. (Optional) Save to Google Drive

Colab sessions are temporary. Save to Google Drive to persist your database.

In [25]:
# Uncomment to save to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('dishdash.db', '/content/drive/MyDrive/dishdash.db')
# print("Database saved to Google Drive!")

---
## Summary & Next Steps

### What We Did (Milestone 3)
- **3 data sources** pulled via APIs: OSM (Overpass), Google Places, Open Food Facts
- **Merged** into a single DataFrame using fuzzy geo-matching + cuisine-key joins
- **Stored** everything in a SQLite database with 3 tables
- **Bonus**: HuggingFace sentiment analysis on Google reviews

### Milestone 4 Ideas (Minimum Viable Unit)
- Build filter/search logic: by cuisine, price, nutrition, sentiment
- Add more neighborhoods beyond NYU area
- Create visualizations: map of restaurants, nutrition comparisons

### Milestone 5 Ideas (Minimal Frontend)
- Flask or Streamlit web app
- Interactive map (Folium or Mapbox)
- Search bar + filter sidebar
- Restaurant detail pages with nutrition cards